## (0) Settings and Functions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import re
import spacy
import unicodedata

from sklearn.model_selection import (
    StratifiedKFold, RepeatedStratifiedKFold, cross_validate, learning_curve, train_test_split
)
from sklearn.multiclass import OneVsRestClassifier
from sklearn.inspection import permutation_importance

from sklearn.feature_extraction.text import CountVectorizer

from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier, PassiveAggressiveClassifier
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.pipeline import Pipeline
from sklearn.base import clone

In [ ]:
df = pd.read_excel("../../data/03_TweetEmotionsPTBR.xlsx")
df.drop(columns=['alegria',	'tristeza',	'raiva', 'medo', 'nojo', 'surpresa', 'confianca', 'antecipacao'], inplace=True)
print(df.shape)
df.head(3)

In [ ]:
# Pré-processamento
# Tenta importar unidecode; se não houver, usamos unicodedata como fallback
try:
    from unidecode import unidecode as _unidecode
    def normalize_unidecode(text: str) -> str:
        return _unidecode(text)
except Exception:
    def normalize_unidecode(text: str) -> str:
        # Fallback razoável: remove diacríticos
        return ''.join(
            c for c in unicodedata.normalize('NFKD', text)
            if not unicodedata.combining(c)
        )

# Carrega modelo spaCy (desabilita componentes pesados que não usamos)
nlp = spacy.load("pt_core_news_lg", disable=["ner", "parser"])  # mantém tokenização, POS, etc.

# Adiciona stopwords (spaCy já tem por padrão)
STOPWORDS = nlp.Defaults.stop_words - {"nao", "nunca", "jamais"}

def preprocess_text_pt_single(text: str) -> str:
    """Pré-processa um único texto (mesma lógica da função anterior)."""
    if not isinstance(text, str):
        return ""
    # 1. remover quebras e normalizar espaços
    text = re.sub(r'\s+', ' ', text.strip())
    # 2. remover HTML, URLs e emails
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    # 3. normalizar acentos (unidecode ou fallback)
    text = normalize_unidecode(text)
    # 4. remover tokens alfanuméricos tipo Q12, A1, C3
    text = re.sub(r'\b[a-zA-Z]*\d+[a-zA-Z]*\b', ' ', text)
    # 5. remover pontuação solta (mantém hífens e apóstrofos dentro de palavras se existirem)
    text = re.sub(r'[^\w\s\-\']', ' ', text)
    # 6. normalizar espaços de novo
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

def preprocess_series_spacy(texts, batch_size=256, n_process=1):
    """Limpa + lematiza + remove stopwords e pontuação."""
    cleaned = (preprocess_text_pt_single(t) for t in texts)
    results = []
    for doc in nlp.pipe(cleaned, batch_size=batch_size, n_process=n_process):
        tokens = [
            tok.lemma_.lower()
            for tok in doc
            if not tok.is_punct
            and not tok.is_space
            and not tok.like_num
            and tok.text.lower() not in STOPWORDS
            and len(tok.text) > 2
        ]
        results.append(" ".join(tokens))
    return results

df['TEXTO_LIMPO'] = preprocess_series_spacy(df['texto'], batch_size=256, n_process=1)

In [ ]:
def avaliar_cv(model, X_train, y_train, n_splits=5, n_repeats=6):
    rskf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=42)

    # macro: trata todas as classes igualmente (classes raras importam tanto quanto classes frequentes)
    # weighted: melhor se houver desbalanceamento (penaliza menos erros em classes raras)
    scoring = {
        'accuracy': 'accuracy',
        'f1_macro': 'f1_macro',
        'precision_macro': 'precision_macro',
        'recall_macro': 'recall_macro'
    }

    results = cross_validate(
        model,
        X_train,
        y_train,
        cv=rskf,
        scoring=scoring,
        n_jobs=-1
    )

    accuracy_scores = results['test_accuracy']
    precision_scores = results['test_precision_macro']
    recall_scores  = results['test_recall_macro']
    f1_scores = results['test_f1_macro']

    return {
        "Accuracy":  (accuracy_scores.mean(),  accuracy_scores.std(),  accuracy_scores),
        "Precision_macro": (precision_scores.mean(), precision_scores.std(), precision_scores),
        "Recall_macro":  (recall_scores.mean(),  recall_scores.std(),  recall_scores),
        "F1_macro":   (f1_scores.mean(),   f1_scores.std(),   f1_scores),
    }

def plotar_learning_curve(model, X_train, y_train, model_name, cv=5, scoring='f1_macro'):
    train_sizes, train_scores, valid_scores = learning_curve(
        model,
        X_train,
        y_train,
        cv=StratifiedKFold(n_splits=cv, shuffle=True, random_state=42),
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring=scoring
    )

    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    valid_mean = valid_scores.mean(axis=1)
    valid_std = valid_scores.std(axis=1)

    plt.figure(figsize=(8,6))
    plt.plot(train_sizes, train_mean, 'o-', color="blue", label="Training Score")
    plt.plot(train_sizes, valid_mean, 'o-', color="orange", label="Validation Score")

    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color="blue")
    plt.fill_between(train_sizes, valid_mean - valid_std, valid_mean + valid_std, alpha=0.1, color="orange")

    plt.title(f"Learning Curve - {model_name}")
    plt.xlabel("Training Set Size")
    plt.ylabel(scoring)
    plt.grid(True)
    plt.legend()
    plt.show()

def plot_feature_importance_top_k(model, feature_names, k=15, model_name="Model"):
    clf = model[-1] if hasattr(model, "__getitem__") else model

    if not hasattr(model, "feature_importances_"):
        raise ValueError("The model does not have feature_importances_.") 
    
    importances = clf.feature_importances_
    idx_sorted = np.argsort(importances)[::-1][:k]
    
    # Plot 
    plt.figure(figsize=(10, 6)) 
    plt.barh(range(k), importances[idx_sorted][::-1]) 
    plt.yticks(range(k), np.array(feature_names)[idx_sorted][::-1])
    plt.title(f"Top-{k} Feature Importances - {model_name}")
    plt.xlabel("Importance") 
    plt.tight_layout() 
    plt.show()

def plot_permutation_importance_top_k(model, X_val, y_val, feature_names, k=15, model_name="Model", scoring="f1_macro"):
    result = permutation_importance(
        model, X_val, y_val,
        n_repeats=10,
        scoring=scoring,
        random_state=42
    )

    importances = result.importances_mean
    idx_sorted = np.argsort(importances)[::-1][:k]

    plt.figure(figsize=(10, 6))
    plt.barh(range(k), importances[idx_sorted][::-1])
    plt.yticks(range(k), np.array(feature_names)[idx_sorted][::-1])
    plt.title(f"Top-{k} Permutation Importance - {model_name}")
    plt.xlabel(f"Impact on {scoring}")
    plt.tight_layout()
    plt.show()

## (1) Predict Baseline

In [ ]:
X_text = df['TEXTO_LIMPO']
y = df['sentimento']

X_text_train, X_text_aux, y_train, y_aux = train_test_split(
    X_text, y,
    train_size=0.70,
    random_state=42,
    stratify=y
)

X_text_dev, X_text_test, y_dev, y_test = train_test_split(
    X_text_aux, y_aux,
    train_size=0.50,
    random_state=42,
    stratify=y_aux
)

In [ ]:
models_for_testing = {
    "LogisticRegression": LogisticRegression(max_iter=None, random_state=42),
    "RidgeClassifier": RidgeClassifier(random_state=42),
    "SGDClassifier": OneVsRestClassifier(SGDClassifier(random_state=42)),
    "PassiveAggressive": OneVsRestClassifier(PassiveAggressiveClassifier(random_state=42)),
    "MultinomialNB": MultinomialNB(),
    "ComplementNB": ComplementNB(),
    "KNN": KNeighborsClassifier(),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "LinearSVC": OneVsRestClassifier(LinearSVC(random_state=42)),
    "SVC": SVC(random_state=42),
    "RandomForest": RandomForestClassifier(random_state=42),
    "XGBClassifier": XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42,
        verbosity=0
    )
}

results = []

def run_predictions(name, model, X_text_train, y_train):
    print("\n" + "=" * 80)
    print(f"Running model: {name}")
    print("=" * 80)

    pipe = Pipeline([
        ("bow", CountVectorizer()),
        ("model", model)
    ])

    cv_res = avaliar_cv(clone(pipe), X_text_train, y_train, n_splits=5, n_repeats=6)

    plotar_learning_curve(clone(pipe), X_text_train, y_train, model_name=name, cv=5, scoring="f1_macro")

    results.append({
        "Model": name,

        "CV_Accuracy_mean":  cv_res["Accuracy"][0],
        "CV_Accuracy_std":   cv_res["Accuracy"][1],
        "CV_Accuracy_all":   cv_res["Accuracy"][2],

        "CV_Precision_macro_mean":  cv_res["Precision_macro"][0],
        "CV_Precision_macro_std":   cv_res["Precision_macro"][1],
        "CV_Precision_macro_all":   cv_res["Precision_macro"][2],

        "CV_Recall_macro_mean":  cv_res["Recall_macro"][0],
        "CV_Recall_macro_std":   cv_res["Recall_macro"][1],
        "CV_Recall_macro_all":   cv_res["Recall_macro"][2],

        "CV_F1_macro_mean":  cv_res["F1_macro"][0],
        "CV_F1_macro_std":   cv_res["F1_macro"][1],
        "CV_F1_macro_all":   cv_res["F1_macro"][2],
    })

for name, model in models_for_testing.items():
    run_predictions(name, model, X_text_train, y_train)

df_results = pd.DataFrame(results)
best_model = df_results.sort_values(by="CV_F1_macro_mean", ascending=False).iloc[0]["Model"]
print("Best model:", best_model)

In [ ]:
def format_results(df):
    formatted_df = df.copy()

    for metric in ["Accuracy", "Precision_macro", "Recall_macro", "F1_macro"]:
        mean_col = f"CV_{metric}_mean"
        std_col = f"CV_{metric}_std"

        formatted_df[metric] = (
            formatted_df[mean_col].round(4).astype(str)
            + " ± "
            + formatted_df[std_col].round(4).astype(str)
        )

        # Remove as colunas originais
        formatted_df = formatted_df.drop(columns=[mean_col, std_col])

    # reorganiza colunas
    cols = ["Model", "Accuracy", "Precision_Macro", "Recall_Macro", "F1_Macro"]
    return formatted_df[cols]

df_results.to_csv("../../data/out/baseline/baseline_03_TweetEmotionsPTBR_BoW.csv")
formatted_df = format_results(df_results)
formatted_df.style.hide(axis="index")

In [ ]:
pipe_rf = Pipeline([
    ("bow", CountVectorizer()),
    ("model", RandomForestClassifier(random_state=42))
])
pipe_rf.fit(X_text_train, y_train)

feature_names = pipe_rf["bow"].get_feature_names_out()
plot_feature_importance_top_k(pipe_rf["model"], feature_names, k=15, model_name="RandomForest")

pipe_xgb = Pipeline([
    ("bow", CountVectorizer()),
    ("model", XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42,
        verbosity=0
    ))
])

pipe_xgb.fit(X_text_train, y_train)
vectorizer = pipe_xgb.named_steps["bow"]
regressor = pipe_xgb.named_steps["model"]

X_dev_transformed = vectorizer.transform(X_text_dev).toarray()

feature_names = vectorizer.get_feature_names_out()

plot_permutation_importance_top_k(
    regressor,
    X_dev_transformed,
    y_dev,
    feature_names,
    k=15,
    model_name="XGBRegressor",
    scoring="f1_macro"
)